In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import nflreadpy as nfl

import keras
from keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss

keras.utils.set_random_seed(42)
DATA_DIR = Path("predict-nfl-wins-fa26ds4420/Data/data/processed")


train = pd.read_csv(DATA_DIR / "nfl_train.csv")
validation = pd.read_csv(DATA_DIR / "nfl_validation.csv")
test = pd.read_csv(DATA_DIR / "nfl_test.csv")
train_seasons = train["season"].unique()

print("Rows (train / validation / test):", len(train), len(validation), len(test))

In [ ]:
games = nfl.load_schedules(seasons=True).to_pandas()
games = games[(games["game_type"] == "REG") & games["home_score"].notna()].copy()

old_names = {"STL": "LA", "SD": "LAC", "OAK": "LV"}
games["home_team"] = games["home_team"].replace(old_names)
games["away_team"] = games["away_team"].replace(old_names)

print(len(games), "games,", games["season"].min(), "-", games["season"].max())

In [ ]:
cols = ["season", "week", "game_id", "team", "opponent_team", "is_home",
        "points_for", "points_against"]
home = games.rename(columns={"home_team": "team", "away_team": "opponent_team",
                             "home_score": "points_for", "away_score": "points_against"})
away = games.rename(columns={"away_team": "team", "home_team": "opponent_team",
                             "away_score": "points_for", "home_score": "points_against"})

log = pd.concat([home.assign(is_home=1)[cols], away.assign(is_home=0)[cols]])
log = log.sort_values(["team", "season", "week"]).reset_index(drop=True)
log["win"] = (log["points_for"] > log["points_against"]).astype(int)

In [ ]:
tep_features = ["points_for", "points_against", "win", "is_home"]
N_MAX = 16

scaler = StandardScaler().fit(log.loc[log["season"].isin(train_seasons), step_features])
log[step_features] = scaler.transform(log[step_features])

by_team = log.groupby("team")[step_features]
lags = pd.concat([by_team.shift(k).add_suffix(f"_lag{k}") for k in range(1, N_MAX + 1)], axis=1)
log = pd.concat([log, lags], axis=1)

log.loc[log["team"] == "ARI", ["season", "week", "points_for", "points_for_lag1", "points_for_lag2"]].head(6)

In [ ]:
lag_cols = list(lags.columns)
team_hist = log[["game_id", "team"] + lag_cols]
opp_hist = team_hist.rename(columns={"team": "opponent_team", **{c: "opp_" + c for c in lag_cols}})


def attach(split):
    out = (split.merge(team_hist, on=["game_id", "team"], validate="one_to_one")
                .merge(opp_hist, on=["game_id", "opponent_team"], validate="one_to_one"))
    assert len(out) == len(split), "some rows didn't match a game in the schedule"
    return out


train, validation, test = attach(train), attach(validation), attach(test)
y_train, y_val, y_test = train["win"].to_numpy(), validation["win"].to_numpy(), test["win"].to_numpy()


def make_inputs(df, n):
    cols = [f"{side}{f}_lag{k}" for k in range(n, 0, -1)          # oldest game first
            for side in ["", "opp_"] for f in step_features]
    X = df[cols].fillna(0).to_numpy("float32").reshape(len(df), n, 2 * len(step_features))
    home = df[["is_home"]].to_numpy("float32")
    return [X, home]

In [ ]:
def make_model(n, units=16, dropout=0.2):
    keras.utils.set_random_seed(42)
    seq_in = keras.Input(shape=(n, 2 * len(step_features)))
    home_in = keras.Input(shape=(1,))

    x = layers.LSTM(units)(seq_in)
    x = layers.Dropout(dropout)(x)
    x = layers.Concatenate()([x, home_in])
    output = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model([seq_in, home_in], output)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


make_model(8).summary()

In [ ]:
models, histories, rows = {}, {}, []
for n in [4, 8, 16]:
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    models[n] = make_model(n)
    histories[n] = models[n].fit(make_inputs(train, n), y_train,
                                 validation_data=(make_inputs(validation, n), y_val),
                                 epochs=60, batch_size=64, callbacks=[stop], verbose=0)
    val_loss = histories[n].history["val_loss"]
    best = int(np.argmin(val_loss))
    rows.append({"n_games": n, "best_epoch": best + 1, "val_log_loss": val_loss[best],
                 "val_accuracy": histories[n].history["val_accuracy"][best]})

for n, h in histories.items():
    plt.plot(h.history["val_loss"], label=f"N = {n}")
plt.xlabel("epoch")
plt.ylabel("validation log loss")
plt.legend()
plt.show()

tuning = pd.DataFrame(rows).sort_values("val_log_loss").reset_index(drop=True)
tuning.round(4)

In [ ]:
best_n = int(tuning.loc[0, "n_games"])
print("picked N =", best_n)

test["prob"] = models[best_n].predict(make_inputs(test, best_n), verbose=0).ravel()
test["pred"] = (test["prob"] > 0.5).astype(int)

bayes = pd.read_csv(DATA_DIR / "predictions_bayes.csv")[["game_id", "team", "prob"]]
mlp = pd.read_csv(DATA_DIR / "predictions_mlp.csv")[["game_id", "team", "prob"]]
both = (test.merge(bayes, on=["game_id", "team"], suffixes=("", "_bayes"), validate="one_to_one")
            .merge(mlp, on=["game_id", "team"], suffixes=("", "_mlp"), validate="one_to_one"))

comparison = pd.DataFrame([
    {"model": f"LSTM (N = {best_n})", "accuracy": accuracy_score(both["win"], both["prob"] > 0.5),
     "log_loss": log_loss(both["win"], both["prob"])},
    {"model": "Bayesian (head-to-head + home)",
     "accuracy": accuracy_score(both["win"], both["prob_bayes"] > 0.5),
     "log_loss": log_loss(both["win"], both["prob_bayes"])},
    {"model": "MLP", "accuracy": accuracy_score(both["win"], both["prob_mlp"] > 0.5),
     "log_loss": log_loss(both["win"], both["prob_mlp"])},
    {"model": "Always pick the home team",
     "accuracy": accuracy_score(both["win"], both["is_home"]), "log_loss": np.nan},
])
print("test rows:", len(both))
print("same pick as Bayesian:", round(((both["prob"] > 0.5) == (both["prob_bayes"] > 0.5)).mean(), 3))
comparison.round(4)